# Open-source Mistral models in place of Ollama: latency test

Runs the test questions in `notebooks/test_data/test_questions.csv` through the project's real RAG path: question parsing, hybrid retrieval at `FINAL_K`, no reranking yet, the `grounded_v4` prompt and the `GroundedAnswer` schema. The model is an open-source one on Mistral's API instead of the local Ollama model. The notebook times every answer against the target of 10–30 seconds per question and checks the answers roughly.

Nothing under `src/` changes. The one replacement is the model call inside `src/rag/generate.py`, which sends Ollama-only options. The **Mistral call** section below repeats that loop with Mistral in its place.

See `MISTRAL_TEST_OVERVIEW.md` for setup, and `docs/mistral-free-tier-evaluation.md` for the findings so far.

In [ ]:
import csv
import os
import re
import sys
import time
from pathlib import Path
from time import perf_counter

import httpx
import pandas as pd
from IPython.display import display

# The repo root is the folder holding src/ and pytest.ini, found from wherever the kernel started.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir() and (p / "pytest.ini").is_file())
sys.path.insert(0, str(ROOT))

from src.config import load_env
from src.rag.constants import MAX_OUTPUT_TOKENS
from src.rag.generate import _is_json, _prose, _validate
from src.rag.prompt import build_prompt
from src.rag.query import parse_question
from src.rag.records import GroundedAnswer
from src.retrieval.constants import FINAL_K

QUESTIONS_CSV = ROOT / "notebooks" / "test_data" / "test_questions.csv"
RESULTS_DIR = ROOT / "notebooks" / "mistral" / "results"

## Settings

In [ ]:
# Two models per run, as {label: Mistral API model id}. Mistral's free plan served these on 18 September 2026:
#   "ministral-14b-2512"  Ministral 3 14B    Apache 2.0
#   "ministral-8b-2512"   Ministral 3 8B     Apache 2.0
#   "voxtral-small-2507"  Voxtral Small 24B  Apache 2.0
#   "codestral-2508"      Codestral 25.08    open weights, licence restricts commercial use
MODELS = {"Ministral 3 14B": "ministral-14b-2512", "Ministral 3 8B": "ministral-8b-2512"}
LIMIT = None             # e.g. 5, to try the first few questions before the full run
CHAT_VIEW_FOR = 3        # questions printed in full, like the chatbot; the rest get one line per answer
TARGET_S, FAST_S = 30.0, 10.0
MIN_INTERVAL_S = 1.1     # between requests, for the free plan's per-second limit; not counted in any time

## API key

Read from the project's `.env` into this kernel only, and never printed. If `.env` has no `MISTRAL_API_KEY`, you're asked for it with hidden input. It then lasts until the kernel restarts. Never paste the key into a cell.

In [ ]:
load_env()  # the project's .env, into this kernel's environment; prints nothing
if not os.environ.get("MISTRAL_API_KEY"):
    from getpass import getpass
    os.environ["MISTRAL_API_KEY"] = getpass("Mistral API key (hidden, kept in memory only): ").strip()
print("Mistral API key:", "found" if os.environ.get("MISTRAL_API_KEY") else "missing")

## Questions

In [ ]:
with open(QUESTIONS_CSV, encoding="utf-8-sig", newline="") as f:
    sheet = [r for r in csv.DictReader(f) if r["Question"].strip()]

questions, originals = [], {}
for n, r in enumerate(sheet, start=1):
    item = re.search(r"item\s+(\d+[a-z]?)", r["Item"], re.IGNORECASE)
    q = {
        "qid": f"Q{n:02d}",
        "question": r["Question"].strip(),
        "ticker": r["Ticker"].strip().upper(),
        "fiscal_year": int(r["Fiscal Year"]),
        "item_label": r["Item"].strip(),
        "item": item.group(1).upper() if item else None,
        "expected": r["Answer"].strip(),
        "paraphrase": r["Paraphrase"].strip().upper() == "TRUE",
    }
    # A paraphrase rewords the nearest original above it with the same company, year and answer.
    key = (q["ticker"], q["fiscal_year"], q["expected"])
    q["paraphrase_of"] = originals.get(key) if q["paraphrase"] else None
    if not q["paraphrase"]:
        originals[key] = q["qid"]
    questions.append(q)
questions = questions[:LIMIT] if LIMIT else questions

print(f"{len(questions)} questions, {sum(q['paraphrase'] for q in questions)} of them paraphrases")
display(pd.DataFrame(questions)[["qid", "ticker", "fiscal_year", "item_label", "paraphrase_of", "question"]])

## Answer checks

These are rough and automatic. Real correctness is marked by hand in the `manual_correct` column of the results.

- **figure found:** the expected dollar amounts and percentages appear in the answer digit for digit.
- **rounding allowed:** a rounded figure also counts, so "$49.6 billion" matches "$49,584 million" (within 0.1%, or 0.05 points for a percentage).
- **terms mentioned:** for list answers, the share of expected items the answer mentions.
- **figure in passages:** whether retrieval gave the model the expected figure at all.

In [ ]:
FIGURE = re.compile(r"\$\s?\d[\d,]*(?:\.\d+)?|\d[\d,]*(?:\.\d+)?\s?%")
NUMBER = re.compile(r"\d[\d,]*(?:\.\d+)?")
AMOUNT = re.compile(r"(\$?)\s?(\d[\d,]*(?:\.\d+)?)\s*(billion|million|thousand|%|percent)?", re.IGNORECASE)
TO_MILLIONS = {"billion": 1000.0, "million": 1.0, "thousand": 0.001}
STOP = {"the", "and", "for", "its", "their", "our", "with", "from", "that", "this", "which", "other", "including",
        "related", "could", "may", "might", "would", "any", "all", "are", "was", "were", "has", "have", "been",
        "into", "such", "than", "not", "per", "of", "in", "to", "as", "on", "by", "or", "an", "at", "is", "it", "be", "we"}


def plain(number):
    number = number.replace(",", "")
    return number.rstrip("0").rstrip(".") if "." in number else number


def figures(text):
    """The dollar amounts and percentages in a text, as bare numbers."""
    return [plain(NUMBER.search(m).group()) for m in FIGURE.findall(text)]


def figure_found(expected, answer):
    wanted = figures(expected)
    if not wanted:
        return None
    given = {plain(n) for n in NUMBER.findall(answer)}
    return sum(w in given for w in wanted) / len(wanted)


def amounts(text):
    """Dollar amounts, in millions, and percentages; a dollar amount needs a "$" or a million/billion after it."""
    found = []
    for dollar, number, unit in AMOUNT.findall(text):
        value, unit = float(number.replace(",", "")), unit.lower()
        if unit in ("%", "percent"):
            found.append(("%", value))
        elif dollar or unit in TO_MILLIONS:
            found.append(("$", value * TO_MILLIONS.get(unit, 1.0)))
    return found


def figure_found_rounded(expected, answer):
    wanted, given = amounts(expected), amounts(answer)
    if not wanted:
        return None
    def close(kind, value, other_kind, other):
        if kind != other_kind:
            return False
        return abs(value - other) <= 0.05 if kind == "%" else value != 0 and abs(value - other) / abs(value) <= 0.001
    return sum(any(close(k, v, gk, gv) for gk, gv in given) for k, v in wanted) / len(wanted)


def words(text):
    found = set()
    for word in re.findall(r"[a-z0-9]+", text.lower()):
        if len(word) >= 2 and word not in STOP:
            found.add(word[:-1] if len(word) > 4 and word.endswith("s") else word)
    return found


def terms_mentioned(expected, answer):
    """For list answers: an item of one or two words needs all of them, a longer one at least half."""
    if figures(expected):
        return None
    items = [w for w in (words(part) for part in re.split(r",|;|\band\b|\bor\b", expected)) if w]
    if not items:
        return None
    have = words(answer)
    return sum(len(i & have) / len(i) >= (1.0 if len(i) <= 2 else 0.5) for i in items) / len(items)


def figure_in_text(expected, text):
    """Whether every expected figure appears, digit for digit, in a text such as the retrieved passages."""
    wanted = figures(expected)
    if not wanted:
        return None
    digits = re.sub(r"[,\s]", "", text)
    return all(re.search(r"(?<!\d)" + re.escape(w) + r"(?!\d)", digits) for w in wanted)

## Retrieval

Retrieval runs on this computer, as it would on the app's server. Loading the indexes and the first search, which loads the embedding model, happen once at start-up, so they're left out of the per-question times. Each question is retrieved once, and both models get the same passages.

In [ ]:
from src.retrieval.bm25 import BM25Retriever
from src.retrieval.dense import DenseRetriever
from src.retrieval.hybrid import HybridRetriever

started = perf_counter()
hybrid = HybridRetriever(BM25Retriever.load(), DenseRetriever.load())
load_s = perf_counter() - started
started = perf_counter()
hybrid.search(parse_question("What was Apple's total net sales in fiscal year 2024?").to_query(top_k=FINAL_K))
print(f"Start-up: indexes {load_s:.1f} s, first search {perf_counter() - started:.1f} s")

for q in questions:
    t0 = perf_counter()
    q["parsed"] = parse_question(q["question"])
    t1 = perf_counter()
    q["passages"] = list(hybrid.search(q["parsed"].to_query(top_k=FINAL_K)))
    t2 = perf_counter()
    q["prompt"] = build_prompt(q["question"], q["passages"]) if q["passages"] else None
    q["retrieval_s"], q["before_model_s"] = t2 - t1, perf_counter() - t0
    q["section_hit"] = any(p.ticker == q["ticker"] and p.fiscal_year == q["fiscal_year"]
                           and (q["item"] is None or p.item == q["item"]) for p in q["passages"])
    q["figure_in_passages"] = figure_in_text(q["expected"], " ".join(p.text for p in q["passages"]))

table = pd.DataFrame(questions)
print(f"Search median {table.retrieval_s.median():.2f} s. Right section in the top {FINAL_K}: "
      f"{int(table.section_hit.sum())}/{len(table)}. Expected figure in the passages: "
      f"{int(table.figure_in_passages.eq(True).sum())}/{int(table.figure_in_passages.notna().sum())} figure questions")
display(table[["qid", "ticker", "fiscal_year", "item_label", "retrieval_s", "section_hit", "figure_in_passages"]])

## The Mistral call

`ask` is `src.rag.generate.stream` with the Mistral call in place of Ollama's. It uses the same schema, sent as a strict JSON schema so that cited source numbers stay within 1–8, plus the same validation and the same prose rendering. Rate limits (HTTP 429) are waited out and retried. That waiting is recorded separately and isn't counted in the answer's time.

In [ ]:
from langchain_mistralai import ChatMistralAI


class StopRun(RuntimeError):
    """An error no retry fixes: a refused key."""


def chat_model(model_id):
    # Temperature 0 and the same output ceiling as the Ollama path. One attempt per call:
    # a hidden retry would add to the measured time.
    return ChatMistralAI(model=model_id, temperature=0.0, max_tokens=MAX_OUTPUT_TOKENS, max_retries=1, timeout=120)


def text_of(content):
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type", "text") == "text")


def ask(llm, prompt, on_text=None):
    schema = GroundedAnswer.for_sources(prompt.n_sources)
    fmt = {"type": "json_schema",
           "json_schema": {"name": "GroundedAnswer", "schema": schema.model_json_schema(), "strict": True}}
    started = perf_counter()
    raw, shown, merged, first_token, first_words, pieces = "", "", None, None, None, 0
    for chunk in llm.stream(prompt.to_messages(), response_format=fmt):
        merged = chunk if merged is None else merged + chunk
        piece = text_of(chunk.content)
        if not piece:
            continue
        now = perf_counter() - started
        first_token = now if first_token is None else first_token
        pieces += 1
        raw += piece
        prose = _prose(raw, complete=False)
        if len(prose) > len(shown) and prose.startswith(shown):
            first_words = now if first_words is None else first_words
            if on_text:
                on_text(prose[len(shown):])
            shown = prose
    total = perf_counter() - started
    answer, parse_error = _validate(raw, schema)
    if answer:
        text = answer.render()
    else:  # cut off or malformed: keep what could be read, as generate.stream does
        text = _prose(raw, complete=_is_json(raw))
        if not text.startswith(shown):
            text = shown
    if on_text and len(text) > len(shown) and text.startswith(shown):
        on_text(text[len(shown):])
    usage = getattr(merged, "usage_metadata", None) or {}
    meta = getattr(merged, "response_metadata", None) or {}
    writing = total - (first_token or 0.0)
    speed_over = writing if pieces > 1 and writing > 0.05 else total
    return {
        "text": text, "valid_json": answer is not None, "abstained": answer.abstained if answer else None,
        "generation_s": total, "first_words_s": first_words if first_words is not None else total,
        "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"),
        "output_tokens_per_s": usage["output_tokens"] / speed_over if usage.get("output_tokens") else None,
        "truncated": meta.get("finish_reason") == "length",
    }


def call(llm, prompt, on_text=None):
    """ask(), waiting out rate limits. Returns (result, error, seconds waited on rate limits)."""
    waited = 0.0
    for attempt in range(1, 8):
        try:
            return ask(llm, prompt, on_text), "", waited
        except httpx.HTTPStatusError as error:
            status, body = error.response.status_code, error.response.text[:300]
            if status == 401 or (status == 403 and "tier" not in body):
                raise StopRun(f"Mistral refused the API key (HTTP {status}); check MISTRAL_API_KEY in .env") from None
            if status != 429 or attempt == 7 or error.response.headers.get("x-ratelimit-limit-req-minute") == "0":
                return None, f"HTTP {status}: {body}", waited
            try:
                wait = float(error.response.headers.get("retry-after"))
            except (TypeError, ValueError):
                wait = min(60.0, 2.0 ** attempt)
            time.sleep(wait)
            waited += wait
        except httpx.HTTPError as error:
            return None, f"{type(error).__name__}: {error}", waited


last_request = [0.0]


def pause():
    gap = MIN_INTERVAL_S - (perf_counter() - last_request[0])
    if gap > 0:
        time.sleep(gap)
    last_request[0] = perf_counter()


COLUMNS = ["qid", "model", "final_k", "question", "expected", "answer", "manual_correct", "end_to_end_s", "first_words_s",
           "retrieval_s", "generation_s", "input_tokens", "output_tokens", "output_tokens_per_s", "abstained", "valid_json", "truncated",
           "section_hit", "figure_in_passages", "figure_found", "figure_found_rounded", "terms_mentioned",
           "paraphrase_of", "error"]


def make_row(q, label, result, error):
    row = dict.fromkeys(COLUMNS)
    row.update({"qid": q["qid"], "model": label, "final_k": FINAL_K, "question": q["question"], "expected": q["expected"],
                "manual_correct": "", "retrieval_s": q["retrieval_s"], "section_hit": q["section_hit"],
                "figure_in_passages": q["figure_in_passages"], "paraphrase_of": q["paraphrase_of"] or "",
                "error": error})
    if result:
        row.update({k: v for k, v in result.items() if k in row and k != "first_words_s"})
        row.update({
            "answer": result["text"],
            "end_to_end_s": q["before_model_s"] + result["generation_s"],
            "first_words_s": q["before_model_s"] + result["first_words_s"],
            "figure_found": figure_found(q["expected"], result["text"]),
            "figure_found_rounded": figure_found_rounded(q["expected"], result["text"]),
            "terms_mentioned": terms_mentioned(q["expected"], result["text"]),
        })
    return row


def check_note(row):
    if row.get("figure_found_rounded") is not None:
        return f"figure found (rounding allowed): {row['figure_found_rounded']:.0%}"
    if row.get("terms_mentioned") is not None:
        return f"expected terms mentioned: {row['terms_mentioned']:.0%}"
    return ""


def one_line(row):
    head = f"{row['qid']} | {row['model']:<17} | "
    if row["error"]:
        return head + "error: " + row["error"][:120]
    return (head + f"{row['end_to_end_s']:5.1f} s end to end, first words {row['first_words_s']:.1f} s | "
            + check_note(row))

## Check the connection

One tiny request per model checks the key and the model name, and shows the bare round trip to Mistral. A model the plan doesn't serve is skipped with Mistral's reason.

In [ ]:
llms = {}
for label, model_id in MODELS.items():
    llm = chat_model(model_id)
    pause()
    started = perf_counter()
    try:
        llm.invoke([{"role": "user", "content": "Reply with the single word OK."}], max_tokens=5)
    except httpx.HTTPStatusError as error:
        status, body = error.response.status_code, error.response.text[:200]
        if status == 401 or (status == 403 and "tier" not in body):
            raise StopRun(f"Mistral refused the API key (HTTP {status}); check MISTRAL_API_KEY in .env") from None
        print(f"{label} ({model_id}): skipped, HTTP {status}: {body}")
        continue
    print(f"{label} ({model_id}): reachable, round trip {(perf_counter() - started) * 1000:.0f} ms")
    llms[label] = (model_id, llm)

## Run

Both models answer each question back to back, over the same passages. The first few questions print as the chatbot would show them: what the parser read, the sources, then the answer as it's written. The rest get one line per answer. Interrupt the kernel to stop early; finished answers are kept.

In [ ]:
rows = []
try:
    for index, q in enumerate(questions):
        chat = index < CHAT_VIEW_FOR
        if chat:
            print("=" * 100)
            print(f"{q['qid']} · {q['ticker']} FY{q['fiscal_year']} · {q['item_label']}")
            print("You:", q["question"])
            print("Read as:", " · ".join(q["parsed"].describe()))
            for n, p in enumerate(q["passages"], start=1):
                table_tag = " (table)" if p.content_type == "table" else ""
                print(f"  [{n}] {p.company} FY{p.fiscal_year}, Item {p.item}: {p.title}{table_tag}")
            print("Expected:", q["expected"])
        for label, (model_id, llm) in llms.items():
            if q["prompt"] is None:
                result, error, waited = None, "no passages retrieved", 0.0
            else:
                pause()
                if chat:
                    print(f"\n{label}: ", end="")
                result, error, waited = call(llm, q["prompt"], (lambda t: print(t, end="", flush=True)) if chat else None)
                if waited:
                    print(f"   (waited {waited:.1f} s on Mistral's rate limit; not counted)")
            row = make_row(q, label, result, error)
            rows.append(row)
            print(("\n   " if chat else "") + one_line(row))
except KeyboardInterrupt:
    print(f"Stopped early; keeping the {len(rows)} answers that finished.")

## Results

The run is saved as `results/<time>_<model ids>.csv`, one row per answer, with an empty `manual_correct` column to mark by hand. `results/comparison.csv` is then rebuilt from every model's latest run.

In [ ]:
def rounded(value, digits=1):
    return None if pd.isna(value) else round(float(value), digits) if digits else round(float(value))


def column(df, name):
    """A column, or an all-null one when the run file predates it.

    ``input_tokens`` and ``final_k`` arrived with #85, so the September runs
    have neither, and comparison.csv is rebuilt from every run file there is.
    """
    return df[name] if name in df.columns else pd.Series(pd.NA, index=df.index, dtype="object")


def paraphrase_agreement(df):
    """Paraphrase pairs whose automatic result matches the original's."""
    def result(r):
        if r.abstained == True:  # noqa: E712 - also true for numpy booleans read back from CSV
            return "abstained"
        return r.figure_found_rounded if pd.notna(r.figure_found_rounded) else round(float(r.terms_mentioned), 2)
    by_qid = {r.qid: r for r in df.itertuples()}
    pairs = [(by_qid[r.paraphrase_of], r) for r in df.itertuples()
             if isinstance(r.paraphrase_of, str) and r.paraphrase_of in by_qid]
    return f"{sum(result(a) == result(b) for a, b in pairs)}/{len(pairs)}"


def summarize(df):
    """One column per model, in the order they first appear."""
    table = {}
    for label in dict.fromkeys(df.model):
        mine = df[df.model == label]
        ok = mine[mine.error.fillna("") == ""]
        e2e = ok.end_to_end_s
        figure_rows = ok[ok.figure_found.notna()]
        terms = ok.terms_mentioned.dropna()
        table[label] = {
            "Answered without error": f"{len(ok)}/{len(mine)}",
            f"Within {TARGET_S:.0f} s end to end": f"{int((e2e <= TARGET_S).sum())}/{len(ok)}",
            f"Within {FAST_S:.0f} s end to end": f"{int((e2e <= FAST_S).sum())}/{len(ok)}",
            "End to end, median (s)": rounded(e2e.median()),
            "End to end, p90 (s)": rounded(e2e.quantile(0.9)),
            "End to end, slowest (s)": rounded(e2e.max()),
            "First words, median (s)": rounded(ok.first_words_s.median()),
            "Output speed, median (tokens/s)": rounded(ok.output_tokens_per_s.median(), 0),
            "Answer length, median (tokens)": rounded(ok.output_tokens.median(), 0),
            "Prompt length, median (tokens)": rounded(column(ok, "input_tokens").median(), 0),
            "Prompt length, largest (tokens)": rounded(column(ok, "input_tokens").max(), 0),
            "FINAL_K": rounded(column(mine, "final_k").iloc[0], 0) if len(mine) else None,
            "Valid answer JSON": f"{int(ok.valid_json.eq(True).sum())}/{len(ok)}",
            "Cut off at the token limit": int(ok.truncated.eq(True).sum()),
            "Abstained": int(ok.abstained.eq(True).sum()),
            "Right section retrieved": f"{int(ok.section_hit.eq(True).sum())}/{len(ok)}",
            "Figure in retrieved passages": f"{int(figure_rows.figure_in_passages.eq(True).sum())}/{len(figure_rows)}",
            "Figure found, exact digits": f"{int((figure_rows.figure_found == 1).sum())}/{len(figure_rows)}",
            "Figure found, rounding allowed": f"{int((figure_rows.figure_found_rounded == 1).sum())}/{len(figure_rows)}",
            "Expected terms mentioned, mean": f"{terms.mean():.0%}" if len(terms) else None,
            "Paraphrase pairs with the same result": paraphrase_agreement(ok),
        }
    return pd.DataFrame(table)


def write_comparison():
    """Rebuild comparison.csv from each model's latest run file (file names start with the run's time)."""
    runs = sorted(p for p in RESULTS_DIR.glob("*.csv") if p.name != "comparison.csv")
    every = pd.concat([pd.read_csv(p).assign(run=p.stem) for p in runs], ignore_index=True)
    latest = every[every.run == every.groupby("model").run.transform("max")]
    comparison = summarize(latest)
    comparison.to_csv(RESULTS_DIR / "comparison.csv", encoding="utf-8-sig")
    return comparison

In [ ]:
run_file = RESULTS_DIR / (time.strftime("%Y%m%d-%H%M") + "_" + "_".join(m for m, _ in llms.values()) + ".csv")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows, columns=COLUMNS).to_csv(run_file, index=False, encoding="utf-8-sig")
print("Saved", run_file.name)

saved = pd.read_csv(run_file)
summary = summarize(saved)
for label in summary.columns:
    s = summary[label]
    print(f"{label}: {s[f'Within {TARGET_S:.0f} s end to end']} within {TARGET_S:.0f} s, "
          f"median {s['End to end, median (s)']} s, slowest {s['End to end, slowest (s)']} s")
display(summary)

print("Every model's latest run, saved to results/comparison.csv:")
display(write_comparison())

In [ ]:
with pd.option_context("display.max_colwidth", 300, "display.max_rows", 200):
    display(saved[["qid", "model", "end_to_end_s", "abstained", "figure_found_rounded", "terms_mentioned", "expected", "answer"]])

## Reading the results

- **End to end** covers parsing, retrieval and the whole answer. The one-off start-up is left out. Retrieval ran on this computer; the app's server may be faster or slower.
- **Free-plan waits aren't counted:** pacing between requests and rate-limit waits come from the free plan, not the model's speed.
- **The checks are proxies.** Reranking isn't in the path yet, so use `manual_correct` in the run's CSV for the real judgement.